In [1]:
import pandas as pd

In [5]:
df = pd.read_csv('conjunto_de_datos_ageb_14_cpv2020.csv', encoding='latin-1', dtype={'ENTIDAD': str,'MUN': str, 'LOC': str, 'AGEB': str, 'MZA': str})
df.head()

,ENTIDAD,NOM_ENT,MUN,NOM_MUN,LOC,NOM_LOC,AGEB,MZA,POBTOT,POBFEM,...,VPH_TELEF,VPH_CEL,VPH_INTER,VPH_STVP,VPH_SPMVPI,VPH_CVJ,VPH_SINRTV,VPH_SINLTC,VPH_SINCINT,VPH_SINTIC
0,14,Jalisco,000,Total de la entidad Jalisco,0000,Total de la entidad,0000,000,8348151,4249696,...,1009963,2134429,1437473,1199700,573528,375678,56399,96948,760439,17553
1,14,Jalisco,001,Acatic,0000,Total del municipio,0000,000,23175,11792,...,1358,5801,2877,3585,482,464,121,310,3193,40
2,14,Jalisco,001,Acatic,0001,Total de la localidad urbana,0000,000,13033,6697,...,985,3293,1940,2530,387,318,54,161,1545,17
3,14,Jalisco,001,Acatic,0001,Total AGEB urbana,0067,000,1266,639,...,73,313,189,246,33,30,9,13,133,*
4,14,Jalisco,001,Acatic,0001,Acatic,0067,001,21,12,...,*,3,3,3,*,3,0,0,0,0


In [6]:
df = df[df['NOM_LOC'] == 'Total AGEB urbana']
df["CVEGEO"] = df["ENTIDAD"] + df["MUN"] + df["LOC"] + df["AGEB"]
df.head()

,ENTIDAD,NOM_ENT,MUN,NOM_MUN,LOC,NOM_LOC,AGEB,MZA,POBTOT,POBFEM,...,VPH_CEL,VPH_INTER,VPH_STVP,VPH_SPMVPI,VPH_CVJ,VPH_SINRTV,VPH_SINLTC,VPH_SINCINT,VPH_SINTIC,CVEGEO
3,14,Jalisco,001,Acatic,0001,Total AGEB urbana,0067,000,1266,639,...,313,189,246,33,30,9,13,133,*,1400100010067
41,14,Jalisco,001,Acatic,0001,Total AGEB urbana,0071,000,2073,1081,...,545,372,508,75,58,8,37,250,4,1400100010071
82,14,Jalisco,001,Acatic,0001,Total AGEB urbana,0103,000,1112,578,...,264,122,119,27,29,6,12,139,*,1400100010103
117,14,Jalisco,001,Acatic,0001,Total AGEB urbana,0118,000,725,356,...,195,82,129,19,13,6,11,123,*,1400100010118
142,14,Jalisco,001,Acatic,0001,Total AGEB urbana,0122,000,1286,642,...,314,193,241,45,32,3,15,130,0,1400100010122


In [7]:
df.to_csv('ageb_14_cpv2020.csv', index=False)

In [2]:
import pandas as pd

df = pd.read_csv("conjunto_de_datos_ageb_urbana_14_cpv2020.csv", dtype=str, encoding="latin1")

# normalize for inspection
nom = df["NOM_LOC"].fillna("").str.strip().str.lower()

# how many MZA=000 total rows by label variant?
mza000 = df[df['NOM_LOC'] == 'Total AGEB urbana'].copy()
mza000["nom_norm"] = mza000["NOM_LOC"].fillna("").str.strip().str.lower()

print("Total rows with MZA=000:", len(mza000))
print(mza000["nom_norm"].value_counts().head(25))


Total rows with MZA=000: 4894
nom_norm
total ageb urbana    4894
Name: count, dtype: int64


In [12]:
#!pip install geopandas
import geopandas as gpd
import pandas as pd

# load shapes
g_ageb = gpd.read_file(r"C:\\Users\\aguev\\Documents\\Maestria_UDG\\tesis\\gdl\\14_jalisco_2020\\conjunto_de_datos\\14a.shp")

# load your filtered rollup output
df_ageb = pd.read_csv("ageb_14_cpv2020.csv", dtype=str)

shp_keys = set(g_ageb["CVEGEO"].astype(str))
csv_keys = set(df_ageb["CVEGEO"].astype(str))

missing_in_csv = sorted(shp_keys - csv_keys)
extra_in_csv = sorted(csv_keys - shp_keys)

print("AGEBs in shapefile:", len(shp_keys))
print("AGEBs in rollup csv:", len(csv_keys))
print("Missing in CSV:", len(missing_in_csv))
print("Extra in CSV:", len(extra_in_csv))

# Save missing list for documentation
pd.DataFrame({"CVEGEO_missing": missing_in_csv}).to_csv("ageb_missing_in_cpv_rollup.csv", index=False)


AGEBs in shapefile: 4892
AGEBs in rollup csv: 4894
Missing in CSV: 0
Extra in CSV: 2


In [11]:
#import geopandas as gpd

# g_ageb is your GeoDataFrame from 14a.shp
missing_gdf = g_ageb[g_ageb["CVEGEO"].astype(str).isin(missing_in_csv)].copy()

# GeoPackage is the most robust for QGIS
missing_gdf.to_file("ageb_missing_in_cpv_rollup.gpkg", layer="missing_ageb", driver="GPKG")

print("Wrote:", "ageb_missing_in_cpv_rollup.gpkg", "features:", len(missing_gdf))


Wrote: ageb_missing_in_cpv_rollup.gpkg features: 41


In [26]:
import osmnx as ox
import geopandas as gpd

place = "Guadalajara, Jalisco, Mexico"

tags = {
    "railway": ["station", "halt"],
    "light_rail": "yes"
}

stations = ox.features_from_place(place, tags).to_crs("EPSG:4326")

# keep points only
stations = stations[stations.geometry.geom_type.isin(["Point", "MultiPoint"])].copy()
stations["geometry"] = stations["geometry"].apply(
    lambda g: g.representative_point() if g.geom_type != "Point" else g
)

print(len(stations), "light rail station candidates")
stations[["name", "railway", "light_rail", "operator", "network"]]


51 light rail station candidates


name  railway light_rail  \
element id                                                                  
node    340135806                           La Aurora     stop        yes   
        340135807                         San Jacinto     stop        yes   
        340135808                          San Andrés     stop        yes   
        340135813                  Cristobal de Oñate     stop        yes   
        340135815                             Oblatos     stop        yes   
        340135816                 Belisario Dominguez     stop        yes   
        340135817                    San Juan de Dios     stop        yes   
        1430300045                  Plaza Universidad     stop        yes   
        4591349821                          La Aurora     stop        yes   
        4591349834                        San Jacinto     stop        yes   
        4591349840                         San Andrés     stop        yes   
        4591349842                 Cristobal de Oñate     stop        yes   
        4591349843                            Oblatos     stop        yes   
        4591349848                Belisario Dominguez     stop        yes   
        4591349851                   San Juan de Dios     stop        yes   
        4591349873                  Plaza Universidad     stop        yes   
        9499989133                 Division del Norte     stop        yes   
        9499989135                 Division del Norte     stop        yes   
        9499989136                      Avila Camacho     stop        yes   
        9499989138                      Avila Camacho     stop        yes   
        9499989139                          Mezquitán     stop        yes   
        9499989141                          Mezquitán     stop        yes   
        9499989142                            Refugio     stop        yes   
        9499989144                            Refugio     stop        yes   
        9499989145                             Juárez     stop        yes   
        9499989146                             Juárez     stop        yes   
        9499989148                      Mexicaltzingo     stop        yes   
        9499989150                      Mexicaltzingo     stop        yes   
        9499989151                         Washington     stop        yes   
        9499989153                         Washington     stop        yes   
        9499989155                     Santa Filomena     stop        yes   
        9499989156                     Santa Filomena     stop        yes   
        9499989157                   Unidad Deportiva     stop        yes   
        9499989159                   Unidad Deportiva     stop        yes   
        9499989160                           Urdaneta     stop        yes   
        9499989161                           Urdaneta     stop        yes   
        9499989163                        18 de Marzo     stop        yes   
        9499989165                        18 de Marzo     stop        yes   
        9499989166                          Isla Raza     stop        yes   
        9499989168                          Isla Raza     stop        yes   
        9499989169                             Patria     stop        yes   
        9499989171                             Patria     stop        yes   
        9499989172                             España     stop        yes   
        9499989174                             España     stop        yes   
        9499989175   Santuario Mártires de Cristo Rey     stop        yes   
        9499989177   Santuario Mártires de Cristo Rey     stop        yes   
        9593569983                          Juárez II     stop        yes   
        9593569984                          Juárez II     stop        yes   
        9862784266                             Tetlán     stop        yes   
        9862784267                             Tetlán     stop        yes   
        12757315248                     Indep

In [27]:
stations

geometry light_rail  \
element id                                                    
node    340135806    POINT (-103.28557 20.66244)        yes   
        340135807    POINT (-103.29729 20.66385)        yes   
        340135808     POINT (-103.3061 20.66526)        yes   
        340135813    POINT (-103.31339 20.66749)        yes   
        340135815    POINT (-103.32252 20.67026)        yes   
        340135816    POINT (-103.33146 20.67272)        yes   
        340135817    POINT (-103.34041 20.67508)        yes   
        1430300045     POINT (-103.3481 20.6751)        yes   
        4591349821   POINT (-103.28557 20.66248)        yes   
        4591349834   POINT (-103.29728 20.66391)        yes   
        4591349840   POINT (-103.30607 20.66532)        yes   
        4591349842   POINT (-103.31337 20.66754)        yes   
        4591349843    POINT (-103.32251 20.6703)        yes   
        4591349848   POINT (-103.33145 20.67276)        yes   
        4591349851    POINT (-103.3404 20.67511)        yes   
        4591349873    POINT (-103.3481 20.67516)        yes   
        9499989133   POINT (-103.35563 20.70797)        yes   
        9499989135    POINT (-103.3556 20.70798)        yes   
        9499989136   POINT (-103.35501 20.69842)        yes   
        9499989138   POINT (-103.35494 20.69841)        yes   
        9499989139   POINT (-103.35395 20.69147)        yes   
        9499989141   POINT (-103.35389 20.69147)        yes   
        9499989142    POINT (-103.3541 20.68237)        yes   
        9499989144   POINT (-103.35404 20.68236)        yes   
        9499989145   POINT (-103.35472 20.67496)        yes   
        9499989146   POINT (-103.35476 20.67496)        yes   
        9499989148   POINT (-103.35541 20.66691)        yes   
        9499989150    POINT (-103.35536 20.6669)        yes   
        9499989151    POINT (-103.35744 20.6611)        yes   
        9499989153    POINT (-103.3574 20.66108)        yes   
        9499989155    POINT (-103.36363 20.6544)        yes   
        9499989156    POINT (-103.3636 20.65438)        yes   
        9499989157   POINT (-103.36931 20.64723)        yes   
        9499989159   POINT (-103.36928 20.64722)        yes   
        9499989160   POINT (-103.37272 20.64315)        yes   
        9499989161   POINT (-103.37275 20.64317)        yes   
        9499989163   POINT (-103.37701 20.63811)        yes   
        9499989165    POINT (-103.37699 20.6381)        yes   
        9499989166   POINT (-103.38071 20.63269)        yes   
        9499989168   POINT (-103.38068 20.63267)        yes   
        9499989169   POINT (-103.38515 20.62667)        yes   
        9499989171   POINT (-103.38512 20.62666)        yes   
        9499989172   POINT (-103.38928 20.62161)        yes   
        9499989174   POINT (-103.38925 20.62159)        yes   
        9499989175   POINT (-103.39563 20.61383)        yes   
        9499989177    POINT (-103.3956 20.61381)        yes   
        9593569983    POINT (-103.35561 20.6749)        yes   
        9593569984   POINT (-103.35561 20.67486)        yes   
        9862784266   POINT (-103.27596 20.65978)        yes   
        9862784267   POINT (-103.27596 20.65981)        yes   
        12757315248   POINT (-103.3447 20.67097)        NaN   

                                                 name public_transport  \
element id                                                               
node    340135806                           La Aurora    stop_position   
        340135807                         San Jacinto    stop_position   
        340135808                          San Andrés    stop_position   
        340135813                  Cristobal de Oñate    stop_position   
        340135815                             Oblatos    stop_position   
        340135816                 Belisario Dominguez    stop_position   
        340135817                    San Juan de Dios    stop_position   
        1430300045                

In [22]:
stations["network"].unique()


array(['Mi Macro Calzada', nan, 'Roll & Bits', 'Mi Tren'], dtype=object)

In [28]:
stations.to_csv("tren_ligero_stations.csv", index=False)

In [15]:
import osmnx as ox
print(ox.__version__)


2.0.7
